# Highway Networks — Depth Experiments

**Notebook:** `04_depth_experiments.ipynb`  
**Research question:** As the number of internal layers increases, do Highway CNNs remain easier to optimize than ordinary CNNs?

This extends our four-layer CIFAR-10 comparison. **It is a small-scale depth study, not an exact replication of the original paper.** We will train both model families **from scratch** using the same 45k/5k training/validation split and compare training loss, validation accuracy, and parameter counts. The test set will not be used for model selection.

## 1. Hypothesis and experimental design

A convolutional Highway Layer computes:

$$
\boxed{y = H(x)\odot T(x) + x\odot(1-T(x))}
$$

where $H(x)$ is a convolution followed by ReLU, $T(x)$ is a sigmoid convolutional transform gate, and $\odot$ means element-wise multiplication. When $T(x)\approx 0$, then $y\approx x$, so information has a more direct path through the layer.

A Standard CNN block computes:

$$
\boxed{y=\operatorname{ReLU}(\operatorname{Conv}(x))}
$$

**Hypothesis:** At greater depths, a Highway CNN may be easier to optimize than a conventional CNN with the same number of internal blocks.

| Setting | Value |
|---|---|
| Internal block counts | 4, 10, 20 |
| Feature channels | 32 |
| Training / validation images | 45,000 / 5,000 |
| Optimizer | Adam, learning rate 0.001 |
| Batch size | 128 |
| Epochs | 10 by default (adjust for testing) |
| Random seed | 42 |

**Limitations:** Each Highway Layer has two convolutions, so parameter counts and computation differ. Our models also have no intermediate spatial downsampling. This investigates optimization of these particular models—not whether gating alone is responsible for any difference.

## 2. Imports and project setup

Run this notebook from the project `notebooks/` directory (or the project root). Reuse the `src/highway.py` and `src/model.py` files from our training notebook.

In [ ]:
import sys
from pathlib import Path
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir if (working_dir / "src").is_dir() else working_dir.parent
if not (PROJECT_ROOT / "src" / "model.py").is_file():
    raise FileNotFoundError("Cannot find src/model.py. Start Jupyter from notebooks/ or the project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model import HighwayCNN, StandardCNN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", device)

## 3. CIFAR-10 training and validation data

We reproduce the **same split rule as Section 6** of `03_cifar10_training.ipynb`: `torch.randperm(50000, generator=torch.Generator().manual_seed(42))`; the first 5,000 indices are validation, and the remainder are training.

- Training images: random crop, horizontal flip, normalization.
- Validation images: normalization only.
- **No test-set evaluation** during this experiment.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
BATCH_SIZE = 128

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

full_train = torchvision.datasets.CIFAR10(
    root=str(PROJECT_ROOT / "data"), train=True, download=True,
    transform=train_transform,
)
full_val = torchvision.datasets.CIFAR10(
    root=str(PROJECT_ROOT / "data"), train=True, download=False,
    transform=val_transform,
)

split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_train), generator=split_generator).tolist()
val_indices = indices[:5000]
train_indices = indices[5000:]
assert set(val_indices).isdisjoint(train_indices)

train_subset = Subset(full_train, train_indices)
val_subset = Subset(full_val, val_indices)

val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    pin_memory=(device.type == "cuda"),
)

def make_train_loader(seed=SEED):
    # Reinitialize the shuffling generator for each fresh model run.
    return DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=(device.type == "cuda"),
        generator=torch.Generator().manual_seed(seed),
    )

print("Train:", len(train_subset), "Validation:", len(val_subset))
print("Training batches:", len(make_train_loader()))
print("Validation batches:", len(val_loader))

## 4. Initialize and verify models at increasing depths

We interpret **depth** as the number of *internal blocks* (`num_highway_layers` / `num_layers`), not the total number of PyTorch modules or convolutions. We create **fresh models** for every run.

First run a smoke test on random data to confirm that each network outputs 10 logits and supports backpropagation.

In [ ]:
DEPTHS = [4, 10, 20]
CHANNELS = 32
GATE_BIAS = -2.0
EPOCHS = 10  # For a quick pipeline check, set to 1 first; reset to 10 for your final experiment.
LR = 0.001


def build_model(kind, depth):
    if kind == "highway":
        return HighwayCNN(
            num_classes=10, channels=CHANNELS,
            num_highway_layers=depth, gate_bias=GATE_BIAS,
        )
    if kind == "standard":
        return StandardCNN(num_classes=10, channels=CHANNELS, num_layers=depth)
    raise ValueError(f"Unknown model kind: {kind}")


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for depth in DEPTHS:
    for kind in ("standard", "highway"):
        torch.manual_seed(SEED)
        net = build_model(kind, depth).to(device)
        x = torch.randn(2, 3, 32, 32, device=device)
        logits = net(x)
        loss = nn.CrossEntropyLoss()(logits, torch.tensor([0, 1], device=device))
        loss.backward()
        assert logits.shape == (2, 10)
        assert torch.isfinite(logits).all()
        assert all(p.grad is not None for p in net.parameters() if p.requires_grad)
        print(f"{kind:8s} | {depth:2d} blocks | {count_parameters(net):>8,d} parameters | PASS")
        del net, x, logits, loss

## 5. Training and evaluation functions

For every epoch we record the **sample-weighted average cross-entropy** and accuracy. Validation uses `model.eval()` and does not update weights. We also measure wall-clock training time.

**Practical note:** Six training runs at 10 epochs each may take a while, especially at 20 blocks. Start by verifying the notebook with `EPOCHS=1` if you want a fast smoke test; change it back to 10 and rerun with fresh models for your real experiment.

In [ ]:
def train_one_epoch(net, loader, criterion, optimizer):
    net.train()
    loss_sum = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = net(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        batch_n = labels.size(0)
        loss_sum += loss.item() * batch_n
        correct += (logits.argmax(1) == labels).sum().item()
        total += batch_n
    return loss_sum / total, 100 * correct / total


@torch.no_grad()
def evaluate(net, loader, criterion):
    net.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = net(images)
        loss = criterion(logits, labels)
        batch_n = labels.size(0)
        loss_sum += loss.item() * batch_n
        correct += (logits.argmax(1) == labels).sum().item()
        total += batch_n
    return loss_sum / total, 100 * correct / total

## 6. Run the depth comparison

This trains **six fresh models**: Standard and Highway, each at 4, 10, and 20 internal blocks. The training loop stores histories and saves checkpoints in `checkpoints/depth_experiments/`. No test images are used.

If you run the cell again, it will **retrain** all six networks and overwrite their checkpoints.

In [ ]:
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "depth_experiments"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

results = {}
criterion = nn.CrossEntropyLoss()

for depth in DEPTHS:
    for kind in ("standard", "highway"):
        # Fresh initialization; resetting helps make runs reproducible.
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        net = build_model(kind, depth).to(device)
        optimizer = torch.optim.Adam(net.parameters(), lr=LR)
        loader = make_train_loader(seed=SEED)
        history = {"train_loss": [], "train_accuracy": [],
                   "val_loss": [], "val_accuracy": []}
        started = time.perf_counter()
        print(f"\nTraining {kind} CNN — {depth} internal blocks")
        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(net, loader, criterion, optimizer)
            val_loss, val_acc = evaluate(net, val_loader, criterion)
            history["train_loss"].append(train_loss)
            history["train_accuracy"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_acc)
            print(f"  Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} "
                  f"| train acc {train_acc:.2f}% | val loss {val_loss:.4f} "
                  f"| val acc {val_acc:.2f}%")

        elapsed = time.perf_counter() - started
        results[(kind, depth)] = {
            "history": history,
            "parameters": count_parameters(net),
            "seconds": elapsed,
        }
        torch.save({
            "model_state_dict": net.state_dict(), "kind": kind,
            "depth": depth, "channels": CHANNELS, "gate_bias": GATE_BIAS,
            "epochs": EPOCHS, "seed": SEED, "history": history,
            "validation_accuracy": history["val_accuracy"][-1],
        }, checkpoint_dir / f"{kind}_{depth}blocks.pth")
        print(f"  Completed in {elapsed / 60:.1f} minutes; checkpoint saved.")
        del net, optimizer, loader

print("\nFinished all depth experiments!")

## 7. Visualize learning across depths

A strong sign of optimization difficulty would be **persistently high training loss** at greater depth. Validation accuracy alone cannot identify an optimization problem because generalization and model capacity also matter.

In [ ]:
# One separate chart per metric, with all model-depth runs on it.
for metric, ylabel, title in [
    ("train_loss", "Cross-entropy loss", "Training loss vs. depth"),
    ("val_accuracy", "Validation accuracy (%)", "Validation accuracy vs. depth"),
]:
    plt.figure(figsize=(9, 5))
    for depth in DEPTHS:
        for kind in ("standard", "highway"):
            h = results[(kind, depth)]["history"]
            plt.plot(range(1, len(h[metric]) + 1), h[metric], marker="o",
                     label=f"{kind.title()} — {depth} blocks")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(ncol=2)
    plt.tight_layout()
    plt.show()

## 8. Final result summary

This table is generated without pandas. Compare final **training loss** first to assess optimization, then validation accuracy and parameter counts.

In [ ]:
header = f"{'Model':<10} {'Blocks':>6} {'Params':>9} {'Train loss':>11} {'Val loss':>10} {'Val acc':>10} {'Time (min)':>11}"
print(header)
print("-" * len(header))
for depth in DEPTHS:
    for kind in ("standard", "highway"):
        result = results[(kind, depth)]
        h = result["history"]
        print(f"{kind.title():<10} {depth:>6} {result['parameters']:>9,d} "
              f"{h['train_loss'][-1]:>11.4f} {h['val_loss'][-1]:>10.4f} "
              f"{h['val_accuracy'][-1]:>9.2f}% {result['seconds']/60:>11.1f}")

### 8.1 Results and observations (fill after running)

**Questions to answer:**

1. At **4 blocks**, how do training loss and validation accuracy compare?
2. At **10 blocks**, does either architecture improve or become harder to train?
3. At **20 blocks**, does the Standard CNN's training loss stall while the Highway CNN continues to optimize—or do both learn?
4. How do parameter counts and runtime grow with depth?
5. Does the observed pattern support or contradict our hypothesis?

**Conclusion:** _Add observations based on your actual output. Do not assume Highway Networks must win: the experiment is valuable either way._

**Caution:** This single-seed, non-parameter-matched study does not establish a general performance advantage. A more rigorous follow-up would compare multiple seeds, controlled parameter budgets, and potentially gradient norms across depths.